[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/02_pytorch_core_building_blocks.ipynb)

# 02. PyTorch core building blocks

현대 모델의 가장 작은 공통 부품을 짧게 한 번씩 실행한다. 깊게 파지 않고 Linear, Conv, activation, normalization, residual, attention, loss, backward, optimizer step의 전체 흐름을 익힌다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Linear + activation


In [ ]:
x = torch.tensor([[1.0, -1.0, 0.5, 2.0]], device=device)

linear = nn.Linear(4, 4, bias=True).to(device)
with torch.no_grad():
    linear.weight.copy_(torch.eye(4, device=device))
    linear.bias.zero_()

z = linear(x)
h = F.silu(z)

print("linear:", z)
print("SiLU:", h)


## 2. Conv2d


In [ ]:
img = torch.arange(25, dtype=torch.float32, device=device).reshape(1, 1, 5, 5)
conv = nn.Conv2d(1, 2, kernel_size=3, padding=1, bias=False).to(device)

y = conv(img)
print("input shape:", img.shape)
print("output shape:", y.shape)


## 3. Normalization + residual


In [ ]:
x = torch.randn(2, 4, 8, device=device)
norm = nn.LayerNorm(8).to(device)
branch = nn.Linear(8, 8, bias=False).to(device)

y = x + branch(norm(x))

print("input norm:", x.norm().item())
print("output norm:", y.norm().item())


## 4. Minimal attention


In [ ]:
B, T, D, H = 1, 4, 8, 2
x = torch.randn(B, T, D, device=device)

qkv = nn.Linear(D, 3 * D, bias=False).to(device)
q, k, v = qkv(x).chunk(3, dim=-1)

head_dim = D // H
q = q.view(B, T, H, head_dim).transpose(1, 2)
k = k.view(B, T, H, head_dim).transpose(1, 2)
v = v.view(B, T, H, head_dim).transpose(1, 2)

attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
attn = attn.transpose(1, 2).reshape(B, T, D)

print("attention output shape:", attn.shape)


## 5. Loss → backward → optimizer


In [ ]:
model = nn.Linear(4, 2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

x = torch.tensor([[1., 0., -1., 2.]], device=device)
target = torch.tensor([1], device=device)

logits = model(x)
loss = F.cross_entropy(logits, target)

optimizer.zero_grad()
loss.backward()

print("loss:", loss.item())
print("weight grad:\n", model.weight.grad)

optimizer.step()
print("updated weight:\n", model.weight)


## 6. Kernel view of the tiny training step


In [ ]:
def tiny_train_step():
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits, target)
    loss.backward()
    optimizer.step()
    return loss

_ = profile_call("tiny training step", tiny_train_step)


## References and provenance

**[2.1] Residual learning**
- 출처: He et al., Deep Residual Learning for Image Recognition
- 이 노트북에서 가져온 부분: residual branch의 최소 구조

**[2.2] Scaled dot-product attention**
- 출처: Vaswani et al., Attention Is All You Need
- 이 노트북에서 가져온 부분: Q/K/V 기반 attention의 최소 구조
